In [1]:
from src.rag_pipeline import RagPipeline
from src.generator import QwenGenerator
import json

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rag = RagPipeline()
generator = QwenGenerator()

with open("data/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(f"Dataset loaded: {len(qa_data)} Q&A pairs")

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Dataset loaded: 7032 Q&A pairs


In [3]:
def answer(query, rag_enabled=True):

    if rag_enabled:
        retrieved = rag.retrieve(query, top_k=3)
        context = "\n".join(
            f"[{i}] Q: {r['question']}\nA: {r['answer']}"
            for i, r in enumerate(retrieved, 1)
        )[:2500]
        generated = generator.generate(query, context=context, max_tokens=256)
    else:
        retrieved = []
        generated = generator.generate(query, context="", max_tokens=256)

    return {
        "answer": generated,
        "sources": (
            [{"question": r["question"], "score": r["score"]} for r in retrieved]
            if rag_enabled
            else []
        ),
        "query": query,
        "rag_enabled": rag_enabled,
    }

In [4]:
question = "What is a pollution?"

result = answer(question, rag_enabled=True)

print("Query:", result["query"])
print("\nAnswer:")
print(result["answer"])

if result["sources"]:
    print("\nSources:")
    for i, src in enumerate(result["sources"], 1):
        print(f"{i}. {src['question'][:60]}... (relevance: {src['score']:.3f})")

Query: What is a pollution?

Answer:
A pollution is defined as the introduction of contaminants into the natural environment that cause adverse changes. These contaminants can be in the form of any substance or energy and can come from both natural and anthropogenic sources.

Sources:
1. What is pollution?... (relevance: 0.950)
2. What is air pollution?... (relevance: 0.914)
3. What is water pollution?... (relevance: 0.904)
